# 06. MCP 서버 기초 (FastMCP 서버 만들기)

> **왜 MCP를 배워야 하나요?**
>
> 에이전트에게 도구를 제공하는 방식이 프로젝트마다 다르면 코드 재사용이 어려워요. MCP는 도구를 **표준화된 방식**으로 패키징해서 어떤 에이전트에서든 즉시 사용할 수 있게 해줘요. 한 번 만든 MCP 서버는 LangChain, Claude, Codex 등 어떤 프레임워크에서도 동일하게 동작해요.

> 🔑 **비유**: MCP는 **USB 포트**와 같아요. USB 이전에는 프린터, 마우스, 키보드마다 다른 포트를 사용했어요. USB가 나온 후에는 어떤 기기든 하나의 포트로 연결할 수 있게 되었죠. MCP가 AI 도구의 세계에서 같은 역할을 해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. MCP(Model Context Protocol)의 개념과 아키텍처를 설명할 수 있어요
2. FastMCP로 `@mcp.tool()` 데코레이터를 사용해 도구를 정의할 수 있어요
3. **stdio** 전송 방식으로 로컬 MCP 서버를 만들고 실행할 수 있어요
4. **streamable-http** 전송 방식으로 원격 MCP 서버를 만들고 실행할 수 있어요
5. MCP Inspector를 활용해 서버를 테스트하고 디버깅할 수 있어요

## 사전 지식

- Part 05의 `02-Tools-V1.ipynb`에서 배운 `@tool` 데코레이터와 도구 바인딩
- Python 비동기 함수(`async def`, `await`) 기초
- LangChain 도구(Tool) 개념

## MCP란 무엇인가요?

**MCP(Model Context Protocol)**는 AI 애플리케이션이 언어 모델에 도구(Tool)와 컨텍스트를 제공하는 방법을 표준화한 오픈 프로토콜이에요.

기존 방식에서는 각 도구마다 개별적인 연동 코드를 작성해야 했어요. 날씨 API, 데이터베이스, 파일 시스템, 외부 서비스 등을 연결할 때마다 서로 다른 방식으로 구현해야 했죠. MCP는 이 모든 것을 **하나의 표준 인터페이스**로 통합해요.

> 🔑 **핵심 개념**: MCP는 USB 포트와 같아요. USB가 어떤 기기든 하나의 표준 방식으로 컴퓨터에 연결할 수 있게 해주듯, MCP는 어떤 도구든 표준 방식으로 LLM에 연결할 수 있게 해줘요.

### MCP의 주요 특징

| 특징 | 설명 |
|------|------|
| 표준화된 도구 인터페이스 | 어떤 서비스든 동일한 방식으로 도구를 정의하고 노출해요 |
| 다양한 전송 메커니즘 | stdio, HTTP(streamable-http) 등 여러 통신 방식을 지원해요 |
| 동적 도구 검색 | 클라이언트가 런타임에 서버의 도구 목록을 자동으로 발견해요 |
| 언어 독립적 | Python, Node.js, Go 등 어떤 언어로도 서버/클라이언트를 구현할 수 있어요 |

### MCP 생태계

```mermaid
flowchart LR
    A["MCP 서버 레지스트리<br>(Smithery AI)"] -->|"공개된 서버<br>목록"| B
    
    subgraph B["MCP 서버들"]
        S1["날씨 서버<br>(stdio)"]
        S2["시간 서버<br>(HTTP)"]
        S3["Context7 서버<br>(npx)"]
        S4["커스텀 서버<br>(FastMCP)"]
    end
    
    B --> C["MCP 클라이언트<br>(MultiServerMCPClient)"]
    C --> D["LLM 에이전트"]
    D --> E["사용자"]
    
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    
    class A storage
    class S1,S2,S3,S4 process
    class C process
    class D input
    class E output
```

## MCP 전송 방식 비교

MCP 서버는 두 가지 주요 전송(transport) 방식을 지원해요.

```mermaid
flowchart TB
    subgraph STDIO["stdio 전송 방식"]
        direction LR
        C1["MCP 클라이언트"] -->|"stdin/stdout"| S1["서버 프로세스<br>(자동 관리)"]
    end
    
    subgraph HTTP["streamable-http 전송 방식"]
        direction LR
        C2["MCP 클라이언트"] -->|"HTTP POST /mcp"| S2["서버 프로세스<br>(독립 실행)"]
    end
    
    STDIO -->|"특징: 로컬, 클라이언트가 프로세스 관리"| NOTE1[" "]
    HTTP -->|"특징: 원격, 서버가 독립적으로 실행"| NOTE2[" "]
    
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef note fill:#fff3cd,stroke:#ffc107,color:#856404
    
    class C1,C2,S1,S2 process
    class NOTE1,NOTE2 note
```

| 전송 방식 | 사용 시점 | 서버 실행 | 엔드포인트 |
|-----------|----------|----------|----------|
| **stdio** | 로컬 개발, 테스트 | 클라이언트가 자동 시작 | 표준 입출력 |
| **streamable-http** | 원격 서버, 프로덕션 | 별도 프로세스로 독립 실행 | `http://host:port/mcp` |

> 🎯 **강의 포인트**: 두 전송 방식의 차이를 실제로 시연해볼게요. stdio는 클라이언트가 서버 프로세스를 자식 프로세스로 자동 생성하기 때문에, 클라이언트만 실행하면 돼요. HTTP는 서버를 먼저 별도로 띄워야 해요.

## 환경 설정

In [1]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# LangSmith 추적 설정 (선택 사항 - 에이전트 실행 과정을 추적해요)
import os

# LangSmith 추적을 활성화해요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-MCP-Basics"

## 1. FastMCP 소개

**FastMCP**는 Python으로 MCP 서버를 만들 수 있는 고수준 라이브러리예요. FastAPI와 유사한 데코레이터 기반 문법을 사용해서 도구를 아주 간결하게 정의할 수 있어요.

> 🔑 **핵심 개념**: FastMCP 서버의 구조는 딱 세 가지예요:
> 1. `FastMCP("서버이름")` — 서버 인스턴스 생성
> 2. `@mcp.tool()` — 도구 등록 (함수에 붙이는 데코레이터)
> 3. `mcp.run(transport="...")` — 서버 실행

### FastMCP 기본 구조

```python
from mcp.server.fastmcp import FastMCP

# 1. 서버 생성
mcp = FastMCP("서버이름", instructions="서버 설명")

# 2. 도구 정의
@mcp.tool()
async def my_tool(param: str) -> str:
    """도구 설명 (LLM이 이 docstring을 읽고 언제 이 도구를 쓸지 결정해요)"""
    return f"결과: {param}"

# 3. 서버 실행
if __name__ == "__main__":
    mcp.run(transport="stdio")  # 또는 "streamable-http"
```

> 💡 **실무 팁**: `@mcp.tool()` 함수의 **docstring**이 매우 중요해요. LLM은 docstring을 읽고 이 도구를 언제, 어떻게 사용할지 결정하기 때문에, 명확하고 구체적인 설명을 작성하면 도구 선택 정확도가 높아져요.

### FastMCP 설치 확인

In [3]:
# mcp 패키지가 설치되어 있는지 확인해요
# 설치되지 않았다면: pip install mcp
import mcp
from mcp.server.fastmcp import FastMCP
import importlib.metadata

mcp_version = importlib.metadata.version("mcp")
print(f"MCP 패키지 버전: {mcp_version}")
# FastMCP import 성공!

MCP 패키지 버전: 1.27.0


## 2. stdio 전송 방식 — 날씨 서버

첫 번째 MCP 서버를 만들어볼게요. **stdio 방식**은 클라이언트가 서버 Python 파일을 서브프로세스로 자동 실행하고, 표준 입출력(stdin/stdout)을 통해 통신해요.

아래 셀은 날씨 서버 파일을 `servers/01_weather_server.py`에 작성해요.

In [4]:
# ---------------------------------------------------
# 날씨 MCP 서버 코드를 파일에 작성해요 (stdio 방식)
# ---------------------------------------------------
# 이 코드는 servers/01_weather_server.py 파일을 생성해요
# 실제 서버는 이 파일을 직접 실행하거나, 클라이언트가 subprocess로 실행해요

weather_server_code = '''
"""
날씨 정보를 제공하는 MCP 서버 (stdio 전송 방식)

이 서버는 FastMCP를 사용해서 날씨 조회 도구를 제공해요.
stdio 방식으로 실행되어 클라이언트가 subprocess로 관리해요.
"""

from mcp.server.fastmcp import FastMCP

# FastMCP 서버 인스턴스를 생성해요
# instructions는 LLM이 이 서버의 목적을 이해하는 데 도움을 줘요
mcp = FastMCP(
    "Weather",
    instructions="날씨 정보를 제공하는 서버예요. 도시 이름을 주면 현재 날씨를 알려줘요.",
)


@mcp.tool()
async def get_weather(location: str) -> str:
    """지정한 도시의 현재 날씨를 조회해요.

    Args:
        location: 날씨를 조회할 도시 이름 (예: 서울, 부산, Tokyo)

    Returns:
        해당 도시의 현재 날씨 정보 문자열
    """
    # 실제 서비스에서는 날씨 API(OpenWeatherMap 등)를 호출해야 해요
    # 여기서는 교육용으로 시뮬레이션 응답을 반환해요
    return f"It\'s always Sunny in {location}"


if __name__ == "__main__":
    # stdio 전송 방식으로 서버를 시작해요
    # 클라이언트가 이 프로세스의 stdin/stdout을 통해 통신해요
    mcp.run(transport="stdio")
'''

# servers 디렉토리가 없으면 생성해요
import os
os.makedirs("servers", exist_ok=True)

# 파일로 저장해요
with open("servers/01_weather_server.py", "w", encoding="utf-8") as f:
    f.write(weather_server_code.strip())

# 날씨 서버 파일 생성 완료: servers/01_weather_server.py

서버 파일을 직접 읽어서 내용을 확인해볼게요.

In [5]:
# ---------------------------------------------------
# 생성된 서버 파일 내용 확인
# ---------------------------------------------------
with open("servers/01_weather_server.py", "r", encoding="utf-8") as f:
    print(f.read())

"""
날씨 정보를 제공하는 MCP 서버 (stdio 전송 방식)

이 서버는 FastMCP를 사용해서 날씨 조회 도구를 제공해요.
stdio 방식으로 실행되어 클라이언트가 subprocess로 관리해요.
"""

from mcp.server.fastmcp import FastMCP

# FastMCP 서버 인스턴스를 생성해요
# instructions는 LLM이 이 서버의 목적을 이해하는 데 도움을 줘요
mcp = FastMCP(
    "Weather",
    instructions="날씨 정보를 제공하는 서버예요. 도시 이름을 주면 현재 날씨를 알려줘요.",
)


@mcp.tool()
async def get_weather(location: str) -> str:
    """지정한 도시의 현재 날씨를 조회해요.

    Args:
        location: 날씨를 조회할 도시 이름 (예: 서울, 부산, Tokyo)

    Returns:
        해당 도시의 현재 날씨 정보 문자열
    """
    # 실제 서비스에서는 날씨 API(OpenWeatherMap 등)를 호출해야 해요
    # 여기서는 교육용으로 시뮬레이션 응답을 반환해요
    return f"It's always Sunny in {location}"


if __name__ == "__main__":
    # stdio 전송 방식으로 서버를 시작해요
    # 클라이언트가 이 프로세스의 stdin/stdout을 통해 통신해요
    mcp.run(transport="stdio")


### stdio 방식으로 MCP 서버에 연결하기

이제 `MultiServerMCPClient`를 사용해서 방금 만든 날씨 서버에 연결해볼게요.

> 🎯 **강의 포인트**: stdio 방식의 핵심은 `"command"`와 `"args"`예요. 클라이언트가 `uv run python servers/01_weather_server.py` 명령을 자동으로 실행해서 서버 프로세스를 관리해요. 터미널에서 서버를 별도로 켜지 않아도 돼요!

In [6]:
# ---------------------------------------------------
# MCP 클라이언트 관련 패키지를 가져와요
# ---------------------------------------------------
# MultiServerMCPClient: 여러 MCP 서버를 동시에 관리해요
from langchain_mcp_adapters.client import MultiServerMCPClient
from concurrent.futures import ThreadPoolExecutor
import asyncio

def run_async(coro):
    """Jupyter 환경에서 비동기 함수를 실행하는 헬퍼예요.
    
    anyio가 asyncio 이벤트 루프 내부에서 직접 실행될 때 충돌이 발생하므로,
    별도 스레드에서 새 이벤트 루프를 생성해 안전하게 실행해요.
    """
    def run_in_thread():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()
    
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(run_in_thread)
        return future.result()

# MCP 클라이언트 패키지 import 성공!
# run_async() 헬퍼 함수 정의 완료 (anyio 이벤트 루프 충돌 방지)

In [7]:
# ---------------------------------------------------
# stdio 방식으로 날씨 서버에 연결하고 도구 목록 확인
# ---------------------------------------------------
# command: 서버 실행 명령 (uv 또는 python)
# args: 명령 인수
# transport: 전송 방식 ("stdio")

# 날씨 서버 연결 설정을 정의해요
weather_server_config = {
    "weather": {
        "command": "uv",          # uv 패키지 매니저로 실행해요
        "args": ["run", "python", "servers/01_weather_server.py"],
        "transport": "stdio",     # stdio 전송 방식을 사용해요
    }
}

async def check_server_tools(server_config: dict, server_name: str) -> None:
    """MCP 서버에 연결하고 제공하는 도구 목록을 출력해요."""
    # 클라이언트를 생성하고 직접 await로 도구를 가져와요
    client = MultiServerMCPClient(server_config)
    tools = await client.get_tools()
    print(f"\n[{server_name}] 서버 도구 목록:")
    for tool in tools:
        print(f"  - 이름: {tool.name}")
        print(f"    설명: {tool.description[:80]}..." if len(tool.description) > 80 else f"    설명: {tool.description}")

# run_async()로 별도 스레드에서 실행해요 (anyio 충돌 방지)
run_async(check_server_tools(weather_server_config, "날씨"))


[날씨] 서버 도구 목록:
  - 이름: get_weather
    설명: 지정한 도시의 현재 날씨를 조회해요.

Args:
    location: 날씨를 조회할 도시 이름 (예: 서울, 부산, Tokyo)

Retu...


### 날씨 도구를 직접 호출해보기

에이전트를 거치지 않고 MCP 도구를 직접 호출해서 동작을 확인해볼게요.

In [8]:
# ---------------------------------------------------
# MCP 도구 직접 호출 (에이전트 없이 테스트)
# ---------------------------------------------------
async def test_weather_tool() -> None:
    """날씨 MCP 도구를 직접 호출해서 테스트해요."""
    client = MultiServerMCPClient(weather_server_config)
    tools = await client.get_tools()
    weather_tool = tools[0]  # 첫 번째 도구 (get_weather)

    # 도구를 직접 호출해요 (LangChain 도구 형식)
    result = await weather_tool.ainvoke({"location": "서울"})
    print(f"서울 날씨: {result}")

    result = await weather_tool.ainvoke({"location": "부산"})
    print(f"부산 날씨: {result}")

    result = await weather_tool.ainvoke({"location": "제주도"})
    print(f"제주도 날씨: {result}")

run_async(test_weather_tool())

서울 날씨: [{'type': 'text', 'text': "It's always Sunny in 서울", 'id': 'lc_dbb9ce2f-a2bc-4c59-bf1b-a3f3ac117422'}]
부산 날씨: [{'type': 'text', 'text': "It's always Sunny in 부산", 'id': 'lc_ae96e2a0-12a6-4eb8-8b95-456a0831461e'}]
제주도 날씨: [{'type': 'text', 'text': "It's always Sunny in 제주도", 'id': 'lc_ec3f66d5-a389-4347-95db-cbfaff9b53c3'}]


> 💡 **실무 팁**: 도구를 직접 호출하는 방식은 MCP 서버 개발 중 빠른 테스트에 유용해요. 에이전트 전체를 실행하지 않아도 도구가 올바르게 동작하는지 확인할 수 있어요.

## 3. streamable-http 전송 방식 — 시간 서버

두 번째 전송 방식인 **streamable-http**를 사용하는 시간 서버를 만들어볼게요. 이 방식은 서버를 별도의 프로세스로 독립 실행한 뒤, 클라이언트가 HTTP로 연결해요.

> ⚠️ **자주 하는 실수**: HTTP 방식 서버는 반드시 **클라이언트 연결 전에** 서버를 먼저 실행해야 해요. 서버가 실행 중이지 않으면 연결 오류가 발생해요. 아래 셀에서 서버 파일을 만든 다음, **터미널에서 별도로 서버를 실행**해야 해요.

In [9]:
# ---------------------------------------------------
# 시간 MCP 서버 코드를 파일에 작성해요 (streamable-http 방식)
# ---------------------------------------------------
time_server_code = '''
"""
현재 시간을 제공하는 MCP 서버 (streamable-http 전송 방식)

이 서버는 HTTP 엔드포인트를 통해 시간 조회 도구를 제공해요.
독립 프로세스로 실행되며, 클라이언트가 HTTP로 연결해요.

실행 방법:
    uv run python servers/02_time_server.py
    # 또는
    python servers/02_time_server.py

클라이언트 연결 URL: http://127.0.0.1:8002/mcp
"""

from mcp.server.fastmcp import FastMCP
from typing import Optional
import pytz
from datetime import datetime

# FastMCP 서버 인스턴스를 생성해요
# port: HTTP 서버가 사용할 포트 번호예요
mcp = FastMCP(
    "CurrentTime",
    instructions="시간대를 지정하면 현재 시간을 알려주는 서버예요.",
    port=8002,  # HTTP 전송 시 사용할 포트
)


@mcp.tool()
async def get_current_time(timezone: Optional[str] = "Asia/Seoul") -> str:
    """지정한 시간대의 현재 시간을 조회해요.

    Args:
        timezone: 조회할 시간대 (기본값: Asia/Seoul)
                  예: America/New_York, Europe/London, Asia/Tokyo

    Returns:
        해당 시간대의 현재 시간 문자열 (YYYY-MM-DD HH:MM:SS TZ 형식)
    """
    try:
        # pytz로 시간대 객체를 가져와요
        tz = pytz.timezone(timezone)

        # 해당 시간대의 현재 시간을 계산해요
        current_time = datetime.now(tz)

        # 읽기 좋은 형식으로 변환해요
        formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S %Z")

        return f"현재 {timezone} 시간: {formatted_time}"

    except pytz.exceptions.UnknownTimeZoneError:
        return f"오류: '{timezone}'은 올바른 시간대가 아니에요. 예: Asia/Seoul, America/New_York"
    except Exception as e:
        return f"시간 조회 중 오류 발생: {str(e)}"


if __name__ == "__main__":
    # 시간 MCP 서버 시작 중... (포트 8002)
    # 클라이언트 연결 URL: http://127.0.0.1:8002/mcp
    # 종료: Ctrl+C

    # streamable-http 전송 방식으로 서버를 시작해요
    # 이 서버는 독립 프로세스로 계속 실행되어야 해요
    mcp.run(transport="streamable-http")
'''

with open("servers/02_time_server.py", "w", encoding="utf-8") as f:
    f.write(time_server_code.strip())

# 시간 서버 파일 생성 완료: servers/02_time_server.py
print()
# 다음 단계:
#   터미널에서 아래 명령으로 서버를 실행하세요:
#   uv run python servers/02_time_server.py

### HTTP 방식 서버에 연결하기

**먼저 터미널에서 서버를 실행해야 해요:**
```bash
uv run python servers/02_time_server.py
```

서버가 실행된 후 아래 셀을 실행하세요.

> 🔑 **핵심 개념**: HTTP 방식은 `"command"`/`"args"` 대신 `"url"`을 사용해요. 서버 주소를 직접 지정하기 때문에, 원격 서버에도 연결할 수 있어요.

In [10]:
# ---------------------------------------------------
# HTTP 방식으로 시간 서버에 연결하기
# ---------------------------------------------------
# 주의: 아래 코드를 실행하기 전에 터미널에서 서버를 먼저 실행해야 해요!
# 명령: uv run python servers/02_time_server.py

# HTTP 서버 연결 설정
# url: 서버의 MCP 엔드포인트 주소 (기본 경로: /mcp)
# transport: "streamable_http" (밑줄 사용, 서버 코드의 "streamable-http"와 달라요)
time_server_config = {
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",   # 서버 엔드포인트
        "transport": "streamable_http",         # 클라이언트 측 설정 (밑줄!)
    }
}

async def test_time_tool() -> None:
    """시간 MCP 도구를 직접 호출해서 테스트해요."""
    client = MultiServerMCPClient(time_server_config)
    tools = await client.get_tools()

    # [시간 서버] 도구 목록:
    for tool in tools:
        print(f"  - {tool.name}: {tool.description[:60]}...")

    # 기본 시간대 (서울)
    result = await tools[0].ainvoke({"timezone": "Asia/Seoul"})
    print(f"\n서울 시간: {result}")

    # 뉴욕 시간
    result = await tools[0].ainvoke({"timezone": "America/New_York"})
    print(f"뉴욕 시간: {result}")

    # 런던 시간
    result = await tools[0].ainvoke({"timezone": "Europe/London"})
    print(f"런던 시간: {result}")

# 서버가 실행 중인 경우에만 연결돼요
# 서버가 실행되지 않았으면 연결 오류가 발생해요
try:
    run_async(test_time_tool())
except Exception as e:
    print(f"연결 오류: {e}")
    # 터미널에서 서버를 먼저 실행해주세요:
    #   uv run python servers/02_time_server.py

  - get_current_time: 지정한 시간대의 현재 시간을 조회해요.

Args:
    timezone: 조회할 시간대 (기본값: Asi...

서울 시간: [{'type': 'text', 'text': '현재 Asia/Seoul 시간: 2026-05-06 15:45:59 KST', 'id': 'lc_d6960274-d12d-4fff-a25f-ba40e7e8a5bf'}]
뉴욕 시간: [{'type': 'text', 'text': '현재 America/New_York 시간: 2026-05-06 02:45:59 EDT', 'id': 'lc_6852a1f5-9274-41ca-b233-2505f48f1c12'}]
런던 시간: [{'type': 'text', 'text': '현재 Europe/London 시간: 2026-05-06 07:45:59 BST', 'id': 'lc_9317f692-3c03-4759-87a1-2718605ba70c'}]


> ⚠️ **자주 하는 실수**: 클라이언트 설정에서 `"streamable_http"` (밑줄)을 사용해요. 서버 코드의 `mcp.run(transport="streamable-http")` (하이픈)과 헷갈리지 않도록 주의하세요.

## 4. 계산기 서버 — 직접 만들어보기

지금까지 날씨 서버(stdio)와 시간 서버(HTTP)를 만들었어요. 이번에는 여러 도구를 가진 계산기 서버를 만들어볼게요. 하나의 MCP 서버에 여러 도구를 등록하는 방법도 배울 수 있어요.

계산기 서버는 `add`, `subtract`, `multiply`, `divide` 네 개의 수학 연산 도구를 제공해요.

> 🎯 **강의 포인트**: `@mcp.tool()` 데코레이터를 여러 함수에 붙이면 하나의 서버에서 여러 도구를 제공할 수 있어요. 클라이언트는 서버에 연결하면 자동으로 모든 도구 목록을 받아와요.

In [11]:
# ---------------------------------------------------
# 계산기 MCP 서버 코드를 파일에 작성해요
# ---------------------------------------------------
# 여러 도구를 가진 서버 예제예요 (stdio 방식)

calculator_server_code = '''
"""
사칙연산을 제공하는 MCP 서버 (stdio 전송 방식)

이 서버는 4가지 수학 연산 도구를 제공해요:
- add: 덧셈
- subtract: 뺄셈
- multiply: 곱셈
- divide: 나눗셈
"""

from mcp.server.fastmcp import FastMCP

# 계산기 서버 인스턴스를 생성해요
mcp = FastMCP(
    "Calculator",
    instructions="수학 연산을 수행하는 계산기 서버예요. 덧셈, 뺄셈, 곱셈, 나눗셈을 지원해요.",
)


@mcp.tool()
async def add(a: float, b: float) -> str:
    """두 숫자를 더해요.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자

    Returns:
        덧셈 결과 문자열
    """
    result = a + b
    return f"{a} + {b} = {result}"


@mcp.tool()
async def subtract(a: float, b: float) -> str:
    """첫 번째 숫자에서 두 번째 숫자를 빼요.

    Args:
        a: 첫 번째 숫자 (피감수)
        b: 두 번째 숫자 (감수)

    Returns:
        뺄셈 결과 문자열
    """
    result = a - b
    return f"{a} - {b} = {result}"


@mcp.tool()
async def multiply(a: float, b: float) -> str:
    """두 숫자를 곱해요.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자

    Returns:
        곱셈 결과 문자열
    """
    result = a * b
    return f"{a} × {b} = {result}"


@mcp.tool()
async def divide(a: float, b: float) -> str:
    """첫 번째 숫자를 두 번째 숫자로 나눠요.

    Args:
        a: 나뉨수 (분자)
        b: 나누는 수 (분모, 0이 되면 오류 반환)

    Returns:
        나눗셈 결과 문자열. b가 0이면 오류 메시지를 반환해요.
    """
    # 0으로 나누기를 방지해요
    if b == 0:
        return "오류: 0으로 나눌 수 없어요!"
    result = a / b
    return f"{a} ÷ {b} = {result:.4f}"


if __name__ == "__main__":
    # stdio 전송 방식으로 서버를 시작해요
    mcp.run(transport="stdio")
'''

with open("servers/03_calculator_server.py", "w", encoding="utf-8") as f:
    f.write(calculator_server_code.strip())

# 계산기 서버 파일 생성 완료: servers/03_calculator_server.py

### 계산기 서버 도구 목록 확인하기

In [12]:
# ---------------------------------------------------
# 계산기 서버 연결 및 도구 목록 확인
# ---------------------------------------------------
calculator_server_config = {
    "calculator": {
        "command": "uv",
        "args": ["run", "python", "servers/03_calculator_server.py"],
        "transport": "stdio",
    }
}

async def test_calculator_tools() -> None:
    """계산기 MCP 서버의 모든 도구를 테스트해요."""
    client = MultiServerMCPClient(calculator_server_config)
    tools = await client.get_tools()

    # [계산기 서버] 등록된 도구 목록:
    for tool in tools:
        print(f"  - {tool.name}")

    print()

    # 도구 이름으로 찾는 헬퍼 함수예요
    def get_tool(name: str):
        return next(t for t in tools if t.name == name)

    # 덧셈 테스트
    result = await get_tool("add").ainvoke({"a": 15, "b": 7})
    print(f"덧셈 결과: {result}")

    # 뺄셈 테스트
    result = await get_tool("subtract").ainvoke({"a": 100, "b": 42})
    print(f"뺄셈 결과: {result}")

    # 곱셈 테스트
    result = await get_tool("multiply").ainvoke({"a": 6, "b": 7})
    print(f"곱셈 결과: {result}")

    # 나눗셈 테스트
    result = await get_tool("divide").ainvoke({"a": 22, "b": 7})
    print(f"나눗셈 결과: {result}")

    # 0으로 나누기 테스트 (오류 처리)
    result = await get_tool("divide").ainvoke({"a": 10, "b": 0})
    print(f"0으로 나누기: {result}")

run_async(test_calculator_tools())

  - add
  - subtract
  - multiply
  - divide

덧셈 결과: [{'type': 'text', 'text': '15.0 + 7.0 = 22.0', 'id': 'lc_afc1658a-5660-4974-a9ae-fb5c474eeeb6'}]
뺄셈 결과: [{'type': 'text', 'text': '100.0 - 42.0 = 58.0', 'id': 'lc_99e57565-e4c2-47e3-ab76-b0fc8c7437db'}]
곱셈 결과: [{'type': 'text', 'text': '6.0 × 7.0 = 42.0', 'id': 'lc_d3259bd8-537d-4da2-a8d6-4aeeab93b71f'}]
나눗셈 결과: [{'type': 'text', 'text': '22.0 ÷ 7.0 = 3.1429', 'id': 'lc_20e09a77-a85b-4ebc-98c6-2084739c2b7d'}]
0으로 나누기: [{'type': 'text', 'text': '오류: 0으로 나눌 수 없어요!', 'id': 'lc_c470039f-5dc7-49a3-9c32-dc59c806a992'}]


## 5. 여러 서버를 동시에 연결하기

`MultiServerMCPClient`의 가장 강력한 기능은 **여러 서버를 동시에 연결**해서 모든 도구를 하나의 목록으로 통합할 수 있다는 거예요.

> 🔑 **핵심 개념**: 에이전트 입장에서는 날씨 서버에서 온 도구인지, 계산기 서버에서 온 도구인지 구분할 필요가 없어요. `client.get_tools()`를 호출하면 연결된 모든 서버의 도구가 하나의 리스트로 합쳐져요.

In [13]:
# ---------------------------------------------------
# 날씨 + 계산기 두 서버를 동시에 연결하기
# ---------------------------------------------------
# 여러 서버 설정을 하나의 딕셔너리에 합쳐요
multi_server_config = {
    # 날씨 서버 (stdio)
    "weather": {
        "command": "uv",
        "args": ["run", "python", "servers/01_weather_server.py"],
        "transport": "stdio",
    },
    # 계산기 서버 (stdio)
    "calculator": {
        "command": "uv",
        "args": ["run", "python", "servers/03_calculator_server.py"],
        "transport": "stdio",
    },
}

async def test_multi_server() -> None:
    """여러 MCP 서버를 동시에 연결하고 통합된 도구 목록을 확인해요."""
    # 모든 서버의 도구를 한 번에 가져와요
    client = MultiServerMCPClient(multi_server_config)
    tools = await client.get_tools()

    print(f"연결된 서버 수: 2개 (날씨, 계산기)")
    print(f"통합 도구 수: {len(tools)}개")
    print()
    # 통합 도구 목록:
    for i, tool in enumerate(tools, 1):
        print(f"  {i}. {tool.name} — {tool.description[:50]}...")

run_async(test_multi_server())

연결된 서버 수: 2개 (날씨, 계산기)
통합 도구 수: 5개

  1. get_weather — 지정한 도시의 현재 날씨를 조회해요.

Args:
    location: 날씨를 조회할 ...
  2. add — 두 숫자를 더해요.

Args:
    a: 첫 번째 숫자
    b: 두 번째 숫자

R...
  3. subtract — 첫 번째 숫자에서 두 번째 숫자를 빼요.

Args:
    a: 첫 번째 숫자 (피감수)...
  4. multiply — 두 숫자를 곱해요.

Args:
    a: 첫 번째 숫자
    b: 두 번째 숫자

R...
  5. divide — 첫 번째 숫자를 두 번째 숫자로 나눠요.

Args:
    a: 나뉨수 (분자)
    ...


## 6. MCP Inspector — 브라우저에서 서버 테스트하기

**MCP Inspector**는 MCP 서버를 브라우저에서 시각적으로 테스트하고 디버깅할 수 있는 공식 도구예요.

### 사용 방법

터미널에서 다음 명령을 실행하면 Inspector가 시작돼요:

```bash
# npx로 실행해요 (Node.js가 필요해요)
npx @modelcontextprotocol/inspector
```

브라우저에서 `http://localhost:6274`로 접속하면 Inspector UI가 열려요.

### Inspector에서 서버 연결하기

Inspector는 왼쪽 패널에서 전송 방식을 고른 뒤 서버 정보를 입력해요. `Command`와 `Arguments`는 한 줄 명령어가 아니라 **실행 파일과 인자**로 나누어 입력한다는 점이 중요해요.

#### STDIO 서버 연결하기 — `servers/01_weather_server.py`

STDIO 방식은 Inspector가 Python 서버 파일을 자식 프로세스로 직접 실행해요. 서버를 터미널에서 따로 띄우지 않아도 됩니다.

| 항목 | 입력값 |
| --- | --- |
| **Transport Type** | `STDIO` |
| **Command** | `/Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/.venv/bin/python` |
| **Arguments** | `/Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/05_Agent_Development/servers/01_weather_server.py` |

또는 `uv`를 사용해 프로젝트 환경을 명시해도 돼요.

| 항목 | 입력값 |
| --- | --- |
| **Transport Type** | `STDIO` |
| **Command** | `/Users/mhso/.local/bin/uv` |
| **Arguments** | `--project /Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents run python /Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/05_Agent_Development/servers/01_weather_server.py` |

> ⚠️ **자주 하는 실수**: `Command`에 `.venv/bin/python`을 넣었다면 `Arguments`에는 Python 파일 경로만 넣어야 해요. `--project ... run python ...`은 `uv` 전용 인자라서 Python에 전달하면 `Unknown option: --project` 오류가 납니다.

> ⚠️ **경로 주의**: 이 노트북의 서버 파일은 프로젝트 루트의 `servers/`가 아니라 `05_Agent_Development/servers/` 아래에 있어요. Inspector의 현재 작업 디렉터리가 어디인지 헷갈릴 수 있으니, 수업 중에는 위처럼 절대 경로를 쓰면 가장 안정적이에요.

연결되면 상태가 `Connected`로 바뀌고 서버 이름 `Weather`가 표시돼요. 그 다음 **Tools** 탭에서 **List Tools**를 누르면 `get_weather` 도구를 확인하고 직접 실행할 수 있어요.

#### Streamable HTTP 서버 연결하기 — `servers/02_time_server.py`

Streamable HTTP 방식은 Inspector가 서버를 실행하지 않아요. 먼저 시간 서버를 별도 터미널에서 실행해두어야 합니다.

```bash
cd /Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/05_Agent_Development
uv run python servers/02_time_server.py
```

서버가 실행 중이면 Inspector에서 다음처럼 연결해요.

| 항목 | 입력값 |
| --- | --- |
| **Transport Type** | `Streamable HTTP` |
| **URL** | `http://127.0.0.1:8002/mcp` |

연결되면 서버 이름 `CurrentTime`이 표시되고, **Tools** 탭에서 `get_current_time` 도구를 호출할 수 있어요.

#### Proxy token 오류가 보일 때

Inspector가 `Connection Error - Did you add the proxy session token in Configuration?` 또는 `proxy token is correct` 같은 메시지를 보여주면, Inspector를 처음 실행한 터미널에 표시된 URL을 그대로 열어야 해요.

```text
http://localhost:6274/?MCP_PROXY_AUTH_TOKEN=...
```

토큰이 빠진 `http://localhost:6274`만 열면 서버 설정이 맞아도 연결이 실패할 수 있어요.

> 💡 **실무 팁**: MCP Inspector는 서버 개발 초기에 도구가 올바르게 등록되었는지, docstring이 제대로 노출되는지 확인하기에 최적의 도구예요. 에이전트 없이 빠르게 기능을 검증할 수 있어요.

> 🎯 **강의 포인트**: 라이브로 Inspector를 실행해서 날씨 서버의 `get_weather` 도구를 직접 호출해볼게요. `location` 파라미터에 여러 도시 이름을 입력해보면서 서버 응답을 확인해요.

## 7. 실습: 간단한 메모 서버 만들기

지금까지 배운 내용을 바탕으로 직접 MCP 서버를 만들어볼 차례예요. 메모를 저장하고 조회하는 간단한 서버를 만들어봐요.

아래 실습 해설 블록에 코드를 작성해보세요.

In [14]:
# ---------------------------------------------------
# 메모 MCP 서버 구현 예제
# ---------------------------------------------------
# 이 서버는 메모리에 메모를 저장하고 조회하는 기능을 제공해요
# 하나의 서버에 여러 도구를 등록하는 패턴을 연습해요

from mcp.server.fastmcp import FastMCP

# 서버 인스턴스 생성
memo_mcp = FastMCP(
    "MemoServer",
    instructions="메모를 저장하고 조회하는 서버예요."
)

# 메모를 저장할 딕셔너리 (서버 메모리에 유지해요)
_memos: dict[str, str] = {}


@memo_mcp.tool()
async def save_memo(title: str, content: str) -> str:
    """새 메모를 저장해요.

    Args:
        title: 메모 제목 (키로 사용돼요)
        content: 메모 내용

    Returns:
        저장 결과 메시지
    """
    # 메모를 딕셔너리에 저장해요
    _memos[title] = content
    return f"메모 '{title}'이(가) 저장되었어요!"


@memo_mcp.tool()
async def get_memo(title: str) -> str:
    """저장된 메모를 조회해요.

    Args:
        title: 조회할 메모 제목

    Returns:
        메모 내용. 없으면 오류 메시지를 반환해요.
    """
    if title not in _memos:
        return f"오류: '{title}' 제목의 메모가 없어요. 저장된 메모: {list(_memos.keys())}"
    return f"[{title}] {_memos[title]}"


@memo_mcp.tool()
async def list_memos() -> str:
    """저장된 모든 메모 제목을 조회해요.

    Returns:
        저장된 메모 제목 목록 문자열
    """
    if not _memos:
        return "저장된 메모가 없어요."
    return f"저장된 메모 목록: {', '.join(_memos.keys())}"


# 메모 서버 인스턴스 생성 완료!
# 등록된 도구:
#   - save_memo: 메모 저장
#   - get_memo: 메모 조회
#   - list_memos: 전체 목록 조회

In [15]:
# ============================================================
# 실습 해설: 메모 서버 직접 실행 및 테스트
# 힌트: FastMCP 서버를 직접 실행하지 않고 Python에서 도구를 직접 호출할 수 있어요
#       메모 MCP 서버 코드를 파일로 저장하고 MultiServerMCPClient로 연결해보세요
# 예상 결과: 메모 저장 → 조회 → 목록 확인 순서로 테스트가 완료되어야 해요
# ============================================================

# 1단계: 메모 서버 코드를 파일로 저장해요
memo_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MemoServer", instructions="메모를 저장하고 조회하는 서버예요.")
_memos = {}

@mcp.tool()
async def save_memo(title: str, content: str) -> str:
    """새 메모를 저장해요."""
    _memos[title] = content
    return f"메모 \'{title}\'이(가) 저장되었어요!"

@mcp.tool()
async def get_memo(title: str) -> str:
    """저장된 메모를 조회해요."""
    if title not in _memos:
        return f"메모 \'{title}\'이(가) 없어요."
    return f"[{title}] {_memos[title]}"

@mcp.tool()
async def list_memos() -> str:
    """저장된 모든 메모 제목을 조회해요."""
    if not _memos:
        return "저장된 메모가 없어요."
    return f"저장된 메모 목록: {list(_memos.keys())}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# 파일 저장
with open("servers/memo_server.py", "w", encoding="utf-8") as f:
    f.write(memo_server_code.strip())

# 2단계: MultiServerMCPClient로 연결해서 테스트해요
memo_config = {
    "memo": {
        "command": "uv",
        "args": ["run", "python", "servers/memo_server.py"],
        "transport": "stdio",
    }
}

async def test_memo_server() -> None:
    """메모 서버 기능을 순서대로 테스트해요."""
    client = MultiServerMCPClient(memo_config)
    tools = await client.get_tools()
    tool_map = {t.name: t for t in tools}

    # === 메모 서버 테스트 ===
    print(f"도구 수: {len(tools)}개\n")

    # 메모 저장 테스트
    result = await tool_map["save_memo"].ainvoke({"title": "오늘 할 일", "content": "LangGraph MCP 노트북 완성하기"})
    print(f"저장: {result}")

    result = await tool_map["save_memo"].ainvoke({"title": "쇼핑 목록", "content": "우유, 달걀, 빵"})
    print(f"저장: {result}")

    # 목록 조회
    result = await tool_map["list_memos"].ainvoke({})
    print(f"\n{result}")

    # 특정 메모 조회
    result = await tool_map["get_memo"].ainvoke({"title": "오늘 할 일"})
    print(f"\n조회: {result}")

    # 없는 메모 조회
    result = await tool_map["get_memo"].ainvoke({"title": "없는 메모"})
    print(f"없는 메모: {result}")

run_async(test_memo_server())

도구 수: 3개

저장: [{'type': 'text', 'text': "메모 '오늘 할 일'이(가) 저장되었어요!", 'id': 'lc_5399f0fd-c776-4446-a279-ba168a8d6696'}]
저장: [{'type': 'text', 'text': "메모 '쇼핑 목록'이(가) 저장되었어요!", 'id': 'lc_f7e33db8-4047-4de1-9d3e-3793ed67031b'}]

[{'type': 'text', 'text': '저장된 메모가 없어요.', 'id': 'lc_ddd12a83-a657-4de5-b7a4-3ab9d1d4246f'}]

조회: [{'type': 'text', 'text': "메모 '오늘 할 일'이(가) 없어요.", 'id': 'lc_17e1080b-ed60-43f5-a56e-816e26494953'}]
없는 메모: [{'type': 'text', 'text': "메모 '없는 메모'이(가) 없어요.", 'id': 'lc_5a3724e6-03f2-498e-803d-fd931445587d'}]


## 8. 서버 구조 정리

이 노트북에서 만든 세 가지 MCP 서버의 구조를 정리해볼게요.

```mermaid
flowchart LR
    subgraph SERVERS["생성된 MCP 서버"]
        direction TB
        W["01_weather_server.py<br>---<br>transport: stdio<br>도구: get_weather"]
        T["02_time_server.py<br>---<br>transport: streamable-http<br>포트: 8002<br>도구: get_current_time"]
        C["03_calculator_server.py<br>---<br>transport: stdio<br>도구: add, subtract<br>       multiply, divide"]
    end
    
    subgraph CLIENT["클라이언트 설정"]
        direction TB
        CS1["command + args<br>→ subprocess 자동 관리"]
        CS2["url<br>→ HTTP 연결"]
    end
    
    W & C --> CS1
    T --> CS2
    CS1 & CS2 --> M["MultiServerMCPClient<br>.get_tools()"]
    M --> AG["LLM 에이전트<br>(다음 노트북에서)"]
    
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    
    class W,T,C process
    class CS1,CS2 storage
    class M,AG output
```

### 만들어진 서버 파일 목록 확인

In [16]:
# ---------------------------------------------------
# 생성된 서버 파일 목록 확인
# ---------------------------------------------------
import os

servers_dir = "servers"
# 생성된 MCP 서버 파일 목록:
print()

for filename in sorted(os.listdir(servers_dir)):
    if filename.endswith(".py"):
        filepath = os.path.join(servers_dir, filename)
        # 파일 크기를 확인해요
        size = os.path.getsize(filepath)
        print(f"  {filename} ({size} bytes)")


  01_weather_server.py (1236 bytes)
  02_time_server.py (2164 bytes)
  03_calculator_server.py (1916 bytes)
  04_database_server.py (9175 bytes)
  05_rag_server.py (7739 bytes)
  06_web_search_server.py (7134 bytes)
  memo_server.py (844 bytes)


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **MCP(Model Context Protocol)**: AI 애플리케이션에서 도구와 컨텍스트를 표준화하는 오픈 프로토콜이에요. USB처럼 어떤 도구든 하나의 표준 방식으로 LLM에 연결해요
- **FastMCP**: Python으로 MCP 서버를 빠르게 만들 수 있는 라이브러리예요. `FastMCP()` + `@mcp.tool()` + `mcp.run()` 세 단계로 서버를 만들어요
- **stdio 전송 방식**: 클라이언트가 서버를 서브프로세스로 자동 관리해요. `command`/`args`로 설정하고 로컬 개발에 적합해요
- **streamable-http 전송 방식**: 서버를 독립 프로세스로 실행하고 클라이언트가 HTTP로 연결해요. `url`로 설정하고 원격 서버나 프로덕션 환경에 적합해요
- **MultiServerMCPClient**: 여러 MCP 서버를 동시에 연결하고 모든 도구를 하나의 목록으로 통합해요
- **MCP Inspector**: `npx @modelcontextprotocol/inspector`로 실행하는 브라우저 기반 디버깅 도구예요

### 전송 방식 비교 요약

| 항목 | stdio | streamable-http |
|------|-------|----------------|
| 서버 실행 | 클라이언트가 자동 | 별도 프로세스로 수동 |
| 설정 키 | `command`, `args` | `url` |
| 적합한 환경 | 로컬 개발, 테스트 | 원격 서버, 프로덕션 |
| 클라이언트 설정 | `"transport": "stdio"` | `"transport": "streamable_http"` |

## 다음 노트북 예고

다음 `07-MCP-Agent-Integration.ipynb`에서는 **MultiServerMCPClient와 `create_agent` 연동**을 배워요. 이번 노트북에서 만든 MCP 서버들을 실제 LLM 에이전트에 연결해서, 에이전트가 MCP 도구를 자율적으로 선택하고 사용하는 방법을 학습해요.